In [6]:
import warnings

import os
import sys
import json
import re
import time 
import random
import math
import torch
import pandas as pd
import numpy as np  

import matplotlib.pyplot as plt

from typing import List, Tuple
from sklearn.model_selection import train_test_split

from transformers import pipeline
from transformers import set_seed
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from transformers import DataCollatorForTokenClassification
from datasets import Dataset, DatasetDict
from evaluate import load
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module=".*torch.*")

In [4]:
data = pd.read_json("data/clean_data.json")
data.head()

,message_id,content
0,shackleton-s/sent/1912.,Message-ID: <21013688.1075844564560.JavaMail.e...
1,farmer-d/logistics/1066.,Message-ID: <22688499.1075854130303.JavaMail.e...
2,parks-j/deleted_items/202.,Message-ID: <27817771.1075841359502.JavaMail.e...
3,stokley-c/chris_stokley/iso/client_rep/41.,Message-ID: <10695160.1075858510449.JavaMail.e...
4,germany-c/all_documents/1174.,Message-ID: <27819143.1075853689038.JavaMail.e...


In [71]:

# Load tokenizer and model from the local directory
MODEL_NAME = "models/dslim-bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, local_files_only=True)

print("Model and tokenizer loaded successfully.")
# Define the labels and their corresponding IDs
LABEL2ID = {"O": 0, "PER": 1, "EML": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


#model.config.label2id = LABEL2ID    
#model.config.id2label = ID2LABEL
#model.num_labels = len(LABEL2ID)  


Model and tokenizer loaded successfully.


In [18]:
# Load NER model
ner_pipeline = pipeline("ner", model=MODEL_NAME, aggregation_strategy="simple", batch_size=8, device=0)
#py_sentimiento = pipeline("sentiment-analysis", model="finiteautomata/beto-sentiment-analysis", tokenizer="finiteautomata/beto-sentiment-analysis", batch_size=8, device=device, truncation=True)

def extract_signature(email_body):
    # Step 1: Use NER to extract entities (name, job title)
    ner_results = ner_pipeline(email_body)
    
    # Step 2: Extract PERSON and ORG entities
    person_name = []
    job_title = []
    
    for entity in ner_results:
        if entity["entity_group"] == "PER":
            person_name.append(entity["word"])
        elif entity["entity_group"] in ["ORG", "MISC"]:
            job_title.append(entity["word"])
    
    full_name = " ".join(person_name) if person_name else "None"
    job_title_text = " ".join(job_title) if job_title else "None"
    
    # Step 3: Extract email using regex
    email_match = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", email_body)
    if email_match:
        num_groups = len(email_match.groups())
        if num_groups > 0:
            print(f"Number of groups: {num_groups}, Groups: {email_match.groups()}, Full match: {email_match.group(0)}")  # Output: Number of groups: 2
            email_address = ""
            for group in email_match.groups():
                email_address = email_address + "|" + group
        else:
            email_address = email_match.group(0)
    else:
        email_address = "None"

    # Step 4: Return dictionary output
    # signature_text' and 'sender' columns
    return {
        "signature_text": full_name + "|" + job_title_text + "|" +email_address,
        "sender": email_address,
        "name": full_name,
        "title": job_title_text,
        "email": email_address
    }


Device set to use cuda:0


In [19]:

#df_subset = df

# Apply extraction to the first 10 rows
df_subset = data.iloc[:1000].copy()  # Select first 10 rows
df_subset["signature"] = df_subset["content"].apply(extract_signature)

# Display the output of emails only
email_values = [signature.get("email") for signature in df_subset["signature"]]
# Create a DataFrame with signature_text and sender columns
signature_df = pd.DataFrame(df_subset["signature"].tolist(), columns=["signature_text", "sender"])

# Save the DataFrame to a CSV file
signature_df.to_csv("signature.csv", index=False)
#print(email_values)
unique_emails = set(email_values)
print(unique_emails)

{'tina.rode@enron.com', 'trevor.woods@enron.com', 'd..hogan@enron.com', 'carol.st.@enron.com', 'al@friedwire.com', 'elizabeth.sager@enron.com', 'kimberly.hillis@enron.com', 'j.kaminski@enron.com', 'ken@kdscommunications.com', 'j..kean@enron.com', 'akatz@eei.org', 'melissa.murphy@enron.com', 'peter.meier@neg.pge.com', 'rhonda.denton@enron.com', 'mark.frevert@enron.com', 'customer.care@dynegy.com', 'mzeleanor@juno.com', 'stephanie.sever@enron.com', 'djenergy@dowjones.com', 'aram.sogomonian@pacificorp.com', 'soblander@carrfut.com', 'lara.leibman@enron.com', 'kevin.presto@enron.com', 'xzhang@haas.berkeley.edu', 'nicole.lauzier@enron.com', 'tdickers@westerngas.com', 'fhaskett@rice.edu', 'sean.bolks@dynegy.com', 'alerts@stockselector.com', 'sara.shackleton@enron.com', 'angela.davis@enron.com', 'mark.fisher@enron.com', 'a..shankman@enron.com', 'delene.travella@funb.com', 'linda.lawrence@enron.com', 'center.ets@enron.com', 'mark.haedicke@enron.com', 'rod.hayslett@enron.com', 'mollie.gustafson@

In [20]:

def email_custom_entity(email_body):
    email_address = ""
        
    # Step 3: Extract email using regex
    email_match = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", email_body)
    if email_match:
        num_groups = len(email_match.groups())
        if num_groups > 0:
            for group in email_match.groups():
                email_address = email_address + "|" + group
        else:
            email_address = email_match.group(0)
    else:
        email_address = "None"

    # Step 4: Return dictionary output
    # signature_text' and 'sender' columns
    return email_address


In [72]:
def label_emails_as_eml(text, tokenizer, labels):
    """
    Labels email addresses in the text as the EML entity for NER training.

    Args:
        text (str): The input text.
        tokenizer (AutoTokenizer): The tokenizer used for tokenizing the text.
        labels (dict): A dictionary mapping entity names to label IDs.

    Returns:
        dict: A dictionary containing tokenized input IDs and corresponding labels.
    """
    # Tokenize the text
    tokenized = tokenizer(text, return_offsets_mapping=True, truncation=True, padding="max_length", max_length=512)
    input_ids = tokenized["input_ids"]
    offsets = tokenized["offset_mapping"]

    # Initialize labels for each token as "O" (outside any entity)
    token_labels = ["O"] * len(input_ids)

    # Use regex to find email addresses in the text
    email_matches = re.finditer(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text)

    for match in email_matches:
        start, end = match.span()
        for idx, (token_start, token_end) in enumerate(offsets):
            if token_start >= start and token_end <= end:
                # Label the first token of the email as "B-EML" and the rest as "I-EML"
                token_labels[idx] = "B-EML" if token_start == start else "I-EML"

    # Convert labels to IDs
    label_ids = [labels[label] if label in labels else labels["O"] for label in token_labels]

    return {"input_ids": input_ids, "labels": label_ids}

# Example usage
sample_text = "Please contact us at support@example.com for more information."
labeled_data = label_emails_as_eml(sample_text, tokenizer, LABEL2ID)
print(labeled_data)

{'input_ids': [101, 4203, 3232, 1366, 1120, 1619, 137, 1859, 119, 3254, 1111, 1167, 1869, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [73]:
# Step 1: Prepare the dataset for training
def prepare_dataset(df, tokenizer, labels):
    """
    Prepares the dataset for training by tokenizing the text and labeling entities.

    Args:
        df (pd.DataFrame): The input dataframe containing email content.
        tokenizer (AutoTokenizer): The tokenizer used for tokenizing the text.
        labels (dict): A dictionary mapping entity names to label IDs.

    Returns:
        DatasetDict: A Hugging Face DatasetDict containing train, validation, and test splits.
    """
    # Apply the labeling function to the dataset
    labeled_data = df["content"].apply(lambda x: label_emails_as_eml(x, tokenizer, labels))

    # Convert the labeled data into a Hugging Face Dataset
    dataset = Dataset.from_pandas(pd.DataFrame(labeled_data.tolist()))

    # Split the dataset into train, validation, and test sets
    dataset = dataset.train_test_split(test_size=0.2, seed=42)
    train_val_split = dataset["train"].train_test_split(test_size=0.1, seed=42)
    dataset["train"] = train_val_split["train"]
    dataset["validation"] = train_val_split["test"]

    return dataset

# Prepare the dataset
dataset = prepare_dataset(df_subset, tokenizer, LABEL2ID)

In [74]:
# Step 2: Define the data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

In [75]:
# Step 3: Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",
    #eval_strategy="epoch", # It appears that the latest version of the transformers library has changed from using "evaluation_strategy" to "eval_strategy".
    eval_strategy="steps", # It appears that the latest version of the transformers library has changed from using "evaluation_strategy" to "eval_strategy".
    learning_rate=2e-5,
    #per_device_train_batch_size=8,
    #per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,
    load_best_model_at_end=True, # --load_best_model_at_end requires the save and eval strategy to match, but found 'steps' and 'epoch' respectively.
    metric_for_best_model="f1",
)

In [76]:
# Define the labels and their corresponding IDs
# Step 4: Define the evaluation metric
metric = load("seqeval")

def compute_metrics(pred):
    """
    Computes evaluation metrics for the model.

    Args:
        pred (EvalPrediction): The predictions from the model.

    Returns:
        dict: A dictionary containing evaluation metrics.
    """
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [77]:

# Step 5: Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [78]:
print(dataset["train"].shape)
print(dataset["validation"].shape)
print(dataset["test"].shape)

(720, 2)
(80, 2)
(200, 2)


In [79]:
# Step 6: Train the model
trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=270, training_loss=0.0009630937267232824, metrics={'train_runtime': 139.36, 'train_samples_per_second': 15.499, 'train_steps_per_second': 1.937, 'total_flos': 564436713553920.0, 'train_loss': 0.0009630937267232824, 'epoch': 3.0})

In [80]:
# Get predictions from the model on the test dataset
test_predictions = trainer.predict(dataset["test"])

# Compute metrics using the compute_metrics function
metrics = compute_metrics((test_predictions.predictions, test_predictions.label_ids))

# Display the model's accuracy
print("Model Accuracy on Test Dataset:")
print(f"Precision: {metrics['precision']}")
print(f"Recall: {metrics['recall']}")
print(f"F1 Score: {metrics['f1']}")
print(f"Accuracy: {metrics['accuracy']}")

Model Accuracy on Test Dataset:
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Accuracy: 1.0


In [43]:
# Save the trained model and tokenizer to disk
output_dir = "custom-ner-model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model and tokenizer saved to {output_dir}")

Model and tokenizer saved to custom-ner-model


In [50]:
from transformers import pipeline

# Load the saved model and tokenizer
saved_model_path = "custom-ner-model"
ner_pipeline_saved = pipeline("ner", model=saved_model_path, tokenizer=saved_model_path, device=0)  # device=0 for GPU

print("NER pipeline created successfully using the saved model.")

Device set to use cuda:0


NER pipeline created successfully using the saved model.


In [55]:
# Step 8: Use the model to detect names and emails from signatures for sample test
def detect_name_and_email(signature, ner_pipeline):
    """
    Detects the name and email from a given signature using the NER pipeline.

    Args:
        signature (str): The input signature text.
        ner_pipeline (Pipeline): The NER pipeline.

    Returns:
        dict: A dictionary containing the detected name and email.
    """
    ner_results = ner_pipeline(signature)
    person_name = []
    print(f"NER Results: {len(ner_results)}")
    for entity in ner_results:
        print(f"{entity["entity_group"]}: {entity["word"]} ({entity["score"]})")
        # Check if the entity is a person name
        if entity["entity_group"] == "PER":
            person_name.append(entity["word"])
    full_name = " ".join(person_name) if person_name else "None"

    email_match = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", signature)
    email_address = email_match.group(0) if email_match else "None"

    return {"name": full_name, "email": email_address}

# Example usage
example_signature = "John Doe | Software Engineer | john.doe@example.com"
ner_pipeline = pipeline("ner", model=MODEL_NAME, aggregation_strategy="simple", batch_size=8, device=0)
result = detect_name_and_email(example_signature, ner_pipeline)
print("Detected Name and Email:", result)

Device set to use cuda:0


NER Results: 1
PER: John Doe (0.9725279211997986)
Detected Name and Email: {'name': 'John Doe', 'email': 'john.doe@example.com'}


In [ ]:
# Step 1: Apply the NER pipeline to the entire dataset
def run_ner_on_dataset_1(data, ner_pipeline):
    """
    Runs the NER pipeline on the entire dataset.

    Args:
        data (pd.DataFrame): The input dataset containing email content.
        ner_pipeline (Pipeline): The NER pipeline.

    Returns:
        pd.DataFrame: A DataFrame with extracted entities (name, email, etc.).
    """
    results = []

    for index, row in data.iterrows():
        content = row["content"]
        ner_results = ner_pipeline(content)

        # Extract entities
        person_name = []
        for entity in ner_results:
            if entity["entity_group"] == "PER":
                person_name.append(entity["word"])

        full_name = " ".join(person_name) if person_name else "None"

        email_match = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", content)
        email_address = email_match.group(0) if email_match else "None"

        results.append({"message_id": row["message_id"], "name": full_name, "email": email_address})

    return pd.DataFrame(results)

# Run the NER pipeline on the entire dataset
ner_results_df = run_ner_on_dataset(data, ner_pipeline)

# Save the results to a CSV file
ner_results_df.to_csv("ner_results.csv", index=False)

# Display the first few rows of the results
print(ner_results_df.head())

                                   message_id  name                      email
0                     shackleton-s/sent/1912.  None  sara.shackleton@enron.com
1                    farmer-d/logistics/1066.  None       pat.clynes@enron.com
2                  parks-j/deleted_items/202.  None             knipe3@msn.com
3  stokley-c/chris_stokley/iso/client_rep/41.  None         kalmeida@caiso.com
4               germany-c/all_documents/1174.  None    chris.germany@enron.com


In [56]:
# Step 1: Apply the NER pipeline to the entire dataset
def run_ner_on_dataset(data, ner_pipeline):
    """
    Runs the NER pipeline on the entire dataset.

    Args:
        data (pd.DataFrame): The input dataset containing email content.
        ner_pipeline (Pipeline): The NER pipeline.

    Returns:
        pd.DataFrame: A DataFrame with extracted entities (name, email, etc.).
    """
    results = []

    for index, row in data.iterrows():
        content = row["content"]
        ner_results = ner_pipeline(content)

        # Extract entities
        person_name = []
        person_email_address = []
        for entity in ner_results:
            if entity["entity_group"] == "PER":
                person_name.append(entity["word"])
            elif entity["entity_group"] == "EML":
                person_email_address.append(entity["word"])
        full_name = " ".join(person_name) if person_name else "None"
        email_match = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", content)
        email_address = email_match.group(0) if email_match else "None"


        results.append({"message_id": row["message_id"], "name": full_name, "email": email_address})

    return pd.DataFrame(results)

ner_pipeline = pipeline("ner", model=MODEL_NAME, aggregation_strategy="simple", batch_size=8, device=0)
# Run the NER pipeline on the entire dataset
ner_results_df = run_ner_on_dataset(df_subset, ner_pipeline)

# Save the results to a CSV file
ner_results_df.to_csv("df_subset_ner_results.csv", index=False)

# Display the first few rows of the results
print(ner_results_df.head())

Device set to use cuda:0


                                   message_id  \
0                     shackleton-s/sent/1912.   
1                    farmer-d/logistics/1066.   
2                  parks-j/deleted_items/202.   
3  stokley-c/chris_stokley/iso/client_rep/41.   
4               germany-c/all_documents/1174.   

                                                name  \
0  Sara Shackleton William Bradford Sara Jeff Den...   
1                 Pat C Ai ##mee Lann Dare Darren Ai   
2  Ch ##et Fen Parks Joe Brian Constantine E ##ri...   
3  Almei Stok Chris St Chris Chris Stok Stok Chri...   
4                   Chris Germany Thomas Engel Chris   

                       email  
0  sara.shackleton@enron.com  
1       pat.clynes@enron.com  
2             knipe3@msn.com  
3         kalmeida@caiso.com  
4    chris.germany@enron.com  


In [ ]:
##### Step 1: Run the NER pipeline on the test data
test_data = dataset["test"]
predictions = []
references = []

for example in test_data:
    input_ids = example["input_ids"]
    labels = example["labels"]

    # Decode the input text
    text = tokenizer.decode(input_ids, skip_special_tokens=True)

    # Get NER predictions
    ner_results = ner_pipeline(text)

    # Convert predictions to label format
    pred_labels = ["O"] * len(text.split())
    for entity in ner_results:
        start, end = entity["start"], entity["end"]
        entity_label = entity["entity_group"]
        for i, word in enumerate(text.split()):
            word_start = text.find(word)
            word_end = word_start + len(word)
            if word_start >= start and word_end <= end:
                pred_labels[i] = f"B-{entity_label}" if word_start == start else f"I-{entity_label}"

    # Convert true labels to label format
    true_labels = [ID2LABEL[label] for label in labels if label != -100]

    predictions.append(pred_labels)
    references.append(true_labels)

# Step 2: Evaluate the model's performance
results = metric.compute(predictions=predictions, references=references)
print("Model Performance:")
print(f"Precision: {results['overall_precision']}")
print(f"Recall: {results['overall_recall']}")
print(f"F1 Score: {results['overall_f1']}")
print(f"Accuracy: {results['overall_accuracy']}")